In [ ]:
import whisper

model = whisper.load_model("base")

# Note the 'r' right before the string
video_path = r"D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video\elarna_video_sorted_2020_2021\grade_01\subject_01\01_02_1_01_02.mp4"

result = model.transcribe(video_path)
print("Transcript: ", result["text"])

Transcript:   Убрай день дорогие ребята! Я рада наши встречи! Тема нашего урока, кто такой, экскуссобот. Сегодня вы ответите на вопросы по тексту. Вы выполните слога звуковой аналий слов со звуком «Э». Напишите слоги и слова сострочной и заглавной буквами «Э». Ребята, скажите, вы любите путешествовать. Во время путешествия мы часто ходим на экскурсии. А знаете ли вы, что это такое? Экскурсия – это коллективное посещение определенных объектов спознавательной или научной целью. Сегодня на уроке мы отправимся на экскурсию. Но отправимся не просто так. Что-то посмотреть о зановыми знаниями. Давайте откроем страницы 106 наших учебников и отгадаем загадки. Он весь город нам покажет. Да еще о нем расскажет. По музеям проведет. Человек верно. Экскурсовод, а кто это? Давайте прочитаем определение этого слова. Экскурсовод – специалист по проведению экскурсий. Он должен хорошо знать историю, культуру и географию того места, где проводит экскурсию. А какое первый звук мы слышим? Словие экскурсовод

In [ ]:
print(result["segments"])

[{'id': 0, 'seek': 0, 'start': 0.0, 'end': 4.0, 'text': ' Убрай день дорогие ребята! Я рада наши встречи!', 'tokens': [50364, 6523, 1552, 481, 3183, 13509, 24365, 5299, 37678, 0, 4857, 26622, 386, 36314, 25669, 435, 0, 50564], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.805668016194332, 'no_speech_prob': 0.08750216662883759}, {'id': 1, 'seek': 0, 'start': 4.0, 'end': 8.0, 'text': ' Тема нашего урока, кто такой, экскуссобот.', 'tokens': [50564, 44064, 386, 45309, 1595, 481, 20296, 11, 12278, 13452, 11, 13817, 4218, 7071, 461, 2061, 1631, 13, 50764], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.805668016194332, 'no_speech_prob': 0.08750216662883759}, {'id': 2, 'seek': 0, 'start': 8.0, 'end': 14.0, 'text': ' Сегодня вы ответите на вопросы по тексту.', 'tokens': [50764, 35913, 2840, 25284, 5878, 1470, 48418, 2801, 1069, 34879, 585, 13, 51064], 'temperature': 0.0, 'avg_logprob': -0.3688893610117387, 'compression_ratio': 1.

In [5]:
import pandas as pd

df_audio_bool = pd.read_csv('master_audio_exist.csv')

df_audio_true = df_audio_bool[df_audio_bool['exist_bool']]
print(df_audio_bool['exist_bool'].sum())
print(len(df_audio_true))

2753
2753


Transcribing Elarna videos, with word level timestamps

In [ ]:
import os
import pandas as pd
import whisper
from pathlib import Path

def transcribe_dataset(video_folder_path, output_dir):
    # 1. Setup paths and directories
    video_root = Path(video_folder_path)
    output_root = Path(output_dir)
    segments_dir = output_root / "transcripts_detailed"
    
    segments_dir.mkdir(parents=True, exist_ok=True)
    master_csv_path = output_root / "master_transcripts.csv"
    df_audio_exist = pd.read_csv("master_audio_exist.csv")

    # 2. Load Whisper Model (change to 'medium' or 'large' if needed)
    print("Loading Whisper model...")
    model = whisper.load_model("base", device="cuda", word_times)
    
    # 3. Load existing progress if available (to allow resuming)
    processed_files = set()
    master_data = []
    
    df_audio_bool = pd.read_csv('master_audio_exist.csv')
    df_audio_true = df_audio_bool[df_audio_bool['exist_bool']]



    if master_csv_path.exists():
        print("Found existing master CSV. Loading progress to skip completed videos...")
        try:
            df_existing = pd.read_csv(master_csv_path)
            processed_files = set(df_existing['video_path'].tolist())
            master_data = df_existing.to_dict('records')
        except Exception as e:
            print(f"Could not read existing CSV, starting fresh. Error: {e}")

    # 4. Find all video files recursively (.mp4, .mkv, .avi)
    video_extensions = ["*.mp4", "*.mkv", "*.avi", "*.mov"]
    video_files = [Path(f) for f in df_audio_true['video_path']]

    # for ext in video_extensions:
    #     video_files.extend(list(video_root.rglob(ext)))

        
    print(f"Found {len(video_files)} total videos. {len(processed_files)} already processed.")

    # 5. Loop through each video
    for i, video_path in enumerate(video_files, 1):
        # Convert to string path for compatibility
        p=Path(video_path)
        video_path_str = str(video_path)
        filename = video_path.name
        filename_stem = video_path.stem  # filename without extension
        
        row = df_audio_exist[df_audio_exist["video_path"] == video_path_str]
        if row.empty:
            continue
        
        if (video_path_str in processed_files):
            continue
            
        print(f"[{i}/{len(video_files)}] Transcribing: {filename}")
        
        try:
            # Run transcription
            result = model.transcribe(video_path_str)
            full_text = result["text"].strip()
            
            # --- Save Individual Segment/Timestamp CSV ---
            segment_records = []
            for segment in result["segments"]:
                segment_records.append({
                    "start_time": segment["start"],
                    "end_time": segment["end"],
                    "phrase": segment["text"].strip()
                })
            
            df_segments = pd.DataFrame(segment_records)
            segment_file_path = segments_dir / f"{filename_stem}_segments.csv"
            df_segments.to_csv(segment_file_path, index=False, encoding='utf-8-sig')
            
            # --- Append to Master Data List ---
            master_data.append({
                "filename": filename,
                "video_path": video_path_str,
                "full_transcript": full_text
            })
            
            # Progressively update Master CSV so you don't lose data if it crashes
            df_master = pd.DataFrame(master_data)
            df_master.to_csv(master_csv_path, index=False, encoding='utf-8-sig')
            
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            continue

    print(f"\n🎉 Done! All transcriptions saved to {output_root}")

# --- EXECUTION ---
if __name__ == "__main__":
    # Replace these paths with your real folder paths
    INPUT_FOLDER = r"D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video"
    OUTPUT_FOLDER = r"D:\KRSL_OnlineSchool\Transcription_Outputs"

In [12]:
import torch
print("Can see GPU?", torch.cuda.is_available())

Can see GPU? True


New code with large-v3 model with better accuracy

In [ ]:
import os
import pandas as pd
import whisper
from pathlib import Path

def transcribe_dataset_v2(video_folder_path, output_dir):
    # 1. Setup paths and directories
    video_root = Path(video_folder_path)
    output_root = Path(output_dir)
    segments_dir = output_root / "transcripts_detailed"
    
    segments_dir.mkdir(parents=True, exist_ok=True)
    master_csv_path = output_root / "master_transcripts.csv"
    
    # Read the audio existence map safely
    df_audio_exist = pd.read_csv("master_audio_exist.csv")
    
    # 🔥 FIX FOR LINUX: Convert Windows paths to Linux SSD paths dynamically
    df_audio_exist['video_path'] = df_audio_exist['video_path'].str.replace('D:\\', '/media/nurobot427/T7/', regex=False)
    df_audio_exist['video_path'] = df_audio_exist['video_path'].str.replace('\\', '/', regex=False)
    
    # Filter after converting paths
    df_audio_true = df_audio_exist[df_audio_exist['exist_bool'] == True]

    # 2. Load Whisper Model (Upgraded to large-v3 per blueprint)
    print("Loading Whisper large-v3 model on CUDA...")
    model = whisper.load_model("large-v3", device="cuda")
    
    # 3. Load existing progress if available (to allow resuming)
    processed_files = set()
    master_data = []
    
    if master_csv_path.exists():
        print("Found existing master CSV. Loading progress to skip completed videos...")
        try:
            df_existing = pd.read_csv(master_csv_path)
            processed_files = set(df_existing['video_path'].tolist())
            master_data = df_existing.to_dict('records')
        except Exception as e:
            print(f"Could not read existing CSV, starting fresh. Error: {e}")

    # 4. Extract targeted paths
    video_files = [Path(f) for f in df_audio_true['video_path']]
    print(f"Found {len(video_files)} total valid videos. {len(processed_files)} already processed.")

    # 5. Loop through each video
    for i, video_path in enumerate(video_files, 1):
        video_path_str = str(video_path)
        filename = video_path.name
        filename_stem = video_path.stem
        
        # Verify it exists in our master spreadsheet map
        row = df_audio_exist[df_audio_exist["video_path"] == video_path_str]
        if row.empty:
            continue
        
        if video_path_str in processed_files:
            continue
            
        print(f"[{i}/{len(video_files)}] Transcribing with word-timestamps: {filename}")
        
        try:
            # Run transcription with strict word boundaries and forced language
            result = model.transcribe(
                video_path_str, 
                word_timestamps=True,
                language="ru" # Force 'ru' or 'kk' to stabilize target alignments
            )
            full_text = result["text"].strip()
            
            # --- Save Word-Level Timestamp CSV ---
            word_records = []
            
            for segment in result["segments"]:
                # Check if word-level data exists in the chunk segment
                if "words" in segment:
                    for w in segment["words"]:
                        word_records.append({
                            "word": w["word"].strip(),
                            "start_time": w["start"],
                            "end_time": w["end"],
                            "probability": w["probability"] 
                        })
                else:
                    # Fallback to phrase level if words breakdown isn't populated
                    word_records.append({
                        "word": segment["text"].strip(),
                        "start_time": segment["start"],
                        "end_time": segment["end"],
                        "probability": None
                    })
            
            df_words = pd.DataFrame(word_records)
            segment_file_path = segments_dir / f"{filename_stem}_words.csv"
            df_words.to_csv(segment_file_path, index=False, encoding='utf-8-sig')
            
            # --- Append to Master Data List ---
            master_data.append({
                "filename": filename,
                "video_path": video_path_str,
                "full_transcript": full_text
            })
            
            # Progressively save Master CSV state
            df_master = pd.DataFrame(master_data)
            df_master.to_csv(master_csv_path, index=False, encoding='utf-8-sig')
            
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            continue

    print(f"\n🎉 Done! All word-level transcripts saved to {output_root}")

# --- Execution ---
INPUT_FOLDER = "/media/nurobot427/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/video"
OUTPUT_FOLDER = "/media/nurobot427/T7/KRSL_OnlineSchool/Transcription_Outputs"

Loading Whisper large-v3 model on CUDA...
Found 2753 total valid videos. 0 already processed.
[1/2753] Transcribing with word-timestamps: 01_04_1_03_06.mp4
[2/2753] Transcribing with word-timestamps: 02_02_6_17_02.mp4
[3/2753] Transcribing with word-timestamps: 02_03_1_03_03.mp4
[4/2753] Transcribing with word-timestamps: 02_03_4_05_02.mp4
[5/2753] Transcribing with word-timestamps: 02_03_4_05_03.mp4


KeyboardInterrupt: 

In [9]:
transcribe_dataset_v2(INPUT_FOLDER, OUTPUT_FOLDER)

Loading Whisper large-v3 model on CUDA...
Found 2753 total valid videos. 0 already processed.
[1/2753] Transcribing with word-timestamps: D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video\elarna_video_unsorted\01_04_1_03_06.mp4
❌ Error processing D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video\elarna_video_unsorted\01_04_1_03_06.mp4: Failed to load audio: ffmpeg version 4.2.2 Copyright (c) 2000-2019 the FFmpeg developers
  built with gcc 7.3.0 (crosstool-NG 1.23.0.449-a04d0)
  configuration: --prefix=/home/nurobot427/miniconda3/envs/sign_lang_env --cc=/tmp/build/80754af9/ffmpeg_1587154242452/_build_env/bin/x86_64-conda_cos6-linux-gnu-cc --disable-doc --enable-avresample --enable-gmp --enable-hardcoded-tables --enable-libfreetype --enable-libvpx --enable-pthreads --enable-libopus --enable-postproc --enable-pic --enable-pthreads --enable-shared --enable-static --enable-version3 --enable-zlib --enable-libmp3lame --disable-nonfree --enable-gpl --enable-gnutls --disable-

KeyboardInterrupt: 

Optimized transcription with faster whisper and batching methos for better use of gpu

In [3]:
import os
import pandas as pd
from faster_whisper import WhisperModel, BatchedInferencePipeline
from pathlib import Path

def transcribe_dataset_v3(video_folder_path, output_dir):
    # 1. Setup paths and directories
    video_root = Path(video_folder_path)
    output_root = Path(output_dir)
    segments_dir = output_root / "transcripts_detailed"
    segments_dir.mkdir(parents=True, exist_ok=True)
    master_csv_path = output_root / "master_transcripts.csv"
    
    # Read the audio existence map safely
    df_audio_exist = pd.read_csv("master_audio_exist.csv")
    
    # Fix paths for Linux SSD
    df_audio_exist['video_path'] = df_audio_exist['video_path'].str.replace('D:\\', '/media/nurobot427/T7/', regex=False)
    df_audio_exist['video_path'] = df_audio_exist['video_path'].str.replace('\\', '/', regex=False)
    df_audio_true = df_audio_exist[df_audio_exist['exist_bool'] == True]

    # 2. Load Optimized Faster-Whisper Model with Batching
    print("Loading Faster-Whisper large-v3 model on CUDA with Batching Pipeline...")
    base_model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    # Wrap the base model to enable concurrent chunk processing
    model = BatchedInferencePipeline(model=base_model)
    
    # 3. Load existing progress to prevent duplicate processing
    processed_files = set()
    if master_csv_path.exists():
        print("Found existing master CSV. Loading progress...")
        try:
            df_existing = pd.read_csv(master_csv_path)
            processed_files = set(df_existing['video_path'].tolist())
        except Exception as e:
            print(f"Could not read existing CSV, starting fresh. Error: {e}")

    # 4. Extract targeted paths
    video_files = [Path(f) for f in df_audio_true['video_path']]
    print(f"Found {len(video_files)} total valid videos. {len(processed_files)} already processed.")

    # 5. Loop through each video
    for i, video_path in enumerate(video_files, 1):
        video_path_str = str(video_path)
        filename = video_path.name
        filename_stem = video_path.stem
        
        if video_path_str in processed_files:
            continue
            
        if not video_path.exists():
            print(f"⚠️ File missing on disk: {filename}")
            continue
            
        print(f"[{i}/{len(video_files)}] Transcribing with faster-whisper: {filename}")
        
        try:
            # transcribe returns a generator (segments) and transcription info
            segments, info = model.transcribe(
                video_path_str, 
                word_timestamps=True,
                language="ru",
                batch_size=48,
                vad_filter=True  # 🔥 Added: Skips silences, prevents hallucinations, and boosts speed
            )
            
            word_records = []
            full_text_chunks = []
            
            # Iterating through segments triggers actual processing lazily
            for segment in segments:
                full_text_chunks.append(segment.text)
                
                # Check for word-level tokens
                if segment.words:
                    for w in segment.words:
                        word_records.append({
                            "word": w.word.strip(),
                            "start_time": w.start,
                            "end_time": w.end,
                            "probability": w.probability 
                        })
                else:
                    # Fallback to segment phrase level
                    word_records.append({
                        "word": segment.text.strip(),
                        "start_time": segment.start,
                        "end_time": segment.end,
                        "probability": None
                    })
            
            full_text = " ".join(full_text_chunks).strip()
            
            # --- Save Word-Level Timestamp CSV ---
            # 🔥 Fix: Ensure structural consistency even if word_records is empty
            df_words = pd.DataFrame(word_records, columns=["word", "start_time", "end_time", "probability"])
            segment_file_path = segments_dir / f"{filename_stem}_words.csv"
            df_words.to_csv(segment_file_path, index=False, encoding='utf-8-sig')
            
            # --- Append Row directly to Master CSV (Massive I/O Optimization) ---
            df_master_row = pd.DataFrame([{
                "filename": filename,
                "video_path": video_path_str,
                "full_transcript": full_text
            }])
            
            # Write header ONLY if the file does not exist yet
            write_header = not master_csv_path.exists()
            df_master_row.to_csv(master_csv_path, mode='a', index=False, header=write_header, encoding='utf-8-sig')
            
            processed_files.add(video_path_str)
            
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            continue

    print(f"\n🎉 Done! All optimized transcripts saved to {output_root}")

# --- Execution ---
INPUT_FOLDER = "/media/nurobot427/T7/KRSL_OnlineSchool/KRSL_OnlineSchool_final_dataset/video"
OUTPUT_FOLDER = "/media/nurobot427/T7/KRSL_OnlineSchool/Transcription_Outputs"

transcribe_dataset_v3(INPUT_FOLDER, OUTPUT_FOLDER)

Loading Faster-Whisper large-v3 model on CUDA with Batching Pipeline...
Found existing master CSV. Loading progress...
Found 2753 total valid videos. 1221 already processed.
[1222/2753] Transcribing with faster-whisper: 19_01_6_11_02.mp4
[1223/2753] Transcribing with faster-whisper: 26_01_6_11_02.mp4
[1224/2753] Transcribing with faster-whisper: 27_10_06_11_02.mp4
[1225/2753] Transcribing with faster-whisper: 27_10_06_14_02.mp4
[1226/2753] Transcribing with faster-whisper: 04_06_06_14_05.mp4
[1227/2753] Transcribing with faster-whisper: 04_22_06_14_05.mp4
[1228/2753] Transcribing with faster-whisper: 06_10_6_14_02.mp4
[1229/2753] Transcribing with faster-whisper: 04_13_6_14_05.mp4
[1230/2753] Transcribing with faster-whisper: 04_15_6_14_05.mp4
[1231/2753] Transcribing with faster-whisper: 04_16_6_14_05.mp4
[1232/2753] Transcribing with faster-whisper: 04_20_6_14_05.mp4
[1233/2753] Transcribing with faster-whisper: 04_29_6_14_05.mp4
[1234/2753] Transcribing with faster-whisper: 04_30_6_